# 🎙️ AI Voice Studio - CosyVoice 3.0 & XTTS v2 통합 GPU 서버 (무료)

<a href="https://colab.research.google.com/github/ssss2513-cyber/ai-audio-studio/blob/main/CosyVoice_XTTS_Colab_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
### 💡 지인분들을 위한 3초 안내서
- **내 컴퓨터 그래픽카드 사양 0% 사용**: 구글이 무료로 빌려주는 **16GB VRAM (Tesla T4 GPU)** 슈퍼컴퓨터로 목소리를 복제합니다.
- **누구나 클릭 한 번으로 작동**: 복잡한 설정이나 명령어 입력 없이, 아래의 재생(▶) 버튼만 누르면 바로 접속 링크가 생성됩니다.

### ⚡ 사용 순서
1. 상단 메뉴 **런타임 ➔ 모두 실행** (단축키: `Ctrl + F9`)을 누릅니다.
2. 약 1~2분 뒤 화면 맨 아래에 출력되는 **`🎉 접속 주소: https://...`** 링크를 클릭하면 끝!

In [ ]:
# [1단계] GPU 16GB 환경 확인 및 필수 시스템 패키지 설치
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg sox libsox-dev build-essential python3-dev git-lfs curl wget
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("\n✅ [1/2] 구글 고성능 16GB GPU 확인 및 시스템 환경 준비 완료!")

In [ ]:
# [2단계] 🔥 CosyVoice 3.0 / 2.0 고음질 한국어 복제 서버 구동 (완벽 자동화 패키지)
import os
import time
import subprocess
import re
import sys

# 1. 이전 실행 프로세스 완전 강제 종료
os.system("pkill -9 -f webui.py 2>/dev/null || true")
os.system("pkill -9 -f cloudflared 2>/dev/null || true")
print("⏳ [1/4] 소스코드 준비 중...")
if not os.path.exists("/content/CosyVoice"):
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git /content/CosyVoice

%cd /content/CosyVoice

print("⏳ [2/4] 핵심 인공지능 라이브러리 전체 무결점 설치 중 (약 1분 소요)...")
# 빌드 툴 및 필수 모듈 일괄 설치
!pip install -q --upgrade pip setuptools wheel Cython
!pip install -q "torchaudio<2.9.0" lightning openai-whisper inflect pyworld-prebuilt onnxruntime-gpu HyperPyYAML conformer diffusers hydra-core omegaconf x-transformers wetext modelscope soundfile gradio librosa gdown wget transformers networkx fastapi

# 필수 모듈 자체 검증 및 보완
required_modules = [
    ('lightning', 'lightning'),
    ('whisper', 'openai-whisper'),
    ('inflect', 'inflect'),
    ('soundfile', 'soundfile'),
    ('gradio', 'gradio'),
    ('transformers', 'transformers')
]
for mod, pkg in required_modules:
    try:
        __import__(mod)
    except ImportError:
        print(f"🔧 필수 모듈 자동 보완 중: {pkg}")
        os.system(f"pip install -q {pkg}")

# torchaudio 라이브러리 자체에 info 함수 영구 주입 (site-packages 레벨 패치)
try:
    import torchaudio
    with open(torchaudio.__file__, "r", encoding="utf-8") as tf:
        t_code = tf.read()
    if "_AudioMetaData" not in t_code:
        with open(torchaudio.__file__, "a", encoding="utf-8") as tf:
            tf.write("\nimport soundfile as _sf\nclass _AudioMetaData:\n    def __init__(self, s): self.sample_rate = s\ninfo = lambda p, **kw: _AudioMetaData(_sf.info(p).samplerate)\n")
    print("✅ torchaudio.info 라이브러리 엔진 영구 패치 완료")
except Exception as te:
    print(f"⚠️ 패치 경고 (무시 가능): {te}")

print("✅ 모든 인공지능 및 오디오 모듈 100% 정상 로드 완료!")

print("⏳ [3/4] 사전 학습 AI 모델 다운로드 중 (초고속 전송)...")
from modelscope import snapshot_download
snapshot_download("iic/CosyVoice2-0.5B", local_dir="pretrained_models/CosyVoice2-0.5B")
print("✅ 모델 다운로드 완료!")

# webui.py 코드 무결점 영구 패치 (torchaudio.info 버그 원천 치환 + share=True)
with open("/content/CosyVoice/webui.py", "r", encoding="utf-8") as f:
    w_code = f.read()

if "import soundfile as sf" not in w_code:
    w_code = "import soundfile as sf\n" + w_code

w_code = re.sub(r"torchaudio\.info\(([^)]+)\)\.sample_rate", r"sf.info(\1).samplerate", w_code)
w_code = re.sub(r"demo\.launch\([^)]*\)", "demo.launch(server_name='0.0.0.0', server_port=args.port, share=True, show_error=True)", w_code)

with open("/content/CosyVoice/webui.py", "w", encoding="utf-8") as f:
    f.write(w_code)

print("⏳ [4/4] Cloudflare 및 Gradio 전세계 공개 접속 주소 생성 중...")
tunnel_proc = subprocess.Popen(
    "cloudflared tunnel --url http://127.0.0.1:50000 --logfile /content/cosy_tunnel.log > /dev/null 2>&1",
    shell=True
)

time.sleep(6)
public_url = None
if os.path.exists("/content/cosy_tunnel.log"):
    with open("/content/cosy_tunnel.log", "r") as f:
        for line in f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if m:
                public_url = m.group(0)
                break

print("\n" + "="*65)
if public_url:
    print(f"🎉 👉 복사/접속할 주소: {public_url}")
print("💡 아래에 생성되는 Gradio 라이브 링크(https://...gradio.live)로도 바로 접속 가능합니다!")
print("="*65 + "\n")

# WebUI 실행
!python3 webui.py --port 50000 --model_dir pretrained_models/CosyVoice2-0.5B


In [ ]:
# [3단계] 🦎 XTTS v2 (Coqui) 제로샷 한국어 복제 웹 서버 구동 (선택사항)
import os
import time
import subprocess

%cd /content
print("⏳ XTTS v2 라이브러리 준비 중...")
!pip install -q coqui-tts gradio soundfile

xtts_webui_code = '''import gradio as gr
import torch
import soundfile as sf
import os
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"XTTS v2 로딩 중... (장치: {device})")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("XTTS v2 로딩 완료!")

def clone_voice(ref_audio, text, language):
    if not ref_audio:
        return None, "⚠️ 참조 오디오 파일을 업로드해주세요."
    out_path = "output_xtts.wav"
    tts.tts_to_file(
        text=text,
        speaker_wav=ref_audio,
        language=language,
        file_path=out_path
    )
    return out_path, "✅ 생성 완료!"

demo = gr.Interface(
    fn=clone_voice,
    inputs=[
        gr.Audio(type="filepath", label="참조 음성 파일 (5~10초 한국어/영어 WAV, MP3)"),
        gr.Textbox(label="생성할 대사 내용", value="안녕하세요! XTTS v2 한국어 음성 복제 테스트입니다.", lines=3),
        gr.Dropdown(choices=["ko", "en", "ja", "zh", "es", "fr", "de"], value="ko", label="언어 선택")
    ],
    outputs=[
        gr.Audio(label="복제된 음성 결과"),
        gr.Textbox(label="진행 상태")
    ],
    title="🦎 XTTS v2 제로샷 목소리 복제 스튜디오 (Google Colab GPU)",
    description="구글 코랩 T4 GPU(16GB) 환경에서 구동되는 고속 한국어 음성 복제 서버입니다."
)

demo.launch(server_name="0.0.0.0", server_port=8080, share=True)
'''

with open("/content/xtts_app.py", "w", encoding="utf-8") as f:
    f.write(xtts_webui_code.strip())

!python3 /content/xtts_app.py
